In [13]:
import pennylane as qml
from pennylane import numpy as np

In [14]:
# ============================================
# Device
# ============================================

dev = qml.device("default.qubit", wires=2)

In [15]:
# ============================================
# Hamiltonian
# H = -1/2 I + 3/2 Z0 - 2 Z1 + Z0 Z1
# ============================================

coeffs = [-0.5, 1.5, -2.0, 1.0]

observables = [
    qml.Identity(0),
    qml.PauliZ(0),
    qml.PauliZ(1),
    qml.PauliZ(0) @ qml.PauliZ(1)
]

H = qml.Hamiltonian(coeffs, observables)

# ============================================
# Variational Ansatz
# ============================================

def ansatz(params):

    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)

    qml.CNOT(wires=[0, 1])

# ============================================
# Quantum Node
# ============================================

@qml.qnode(dev)
def circuit(params):
    
    ansatz(params)

    return qml.expval(H)

In [16]:
# ============================================
# Optimization
# ============================================

params = np.random.uniform(0, 2*np.pi, 2, requires_grad=True)

optimizer = qml.GradientDescentOptimizer(stepsize=0.1)

steps = 100

for i in range(steps):

    params = optimizer.step(circuit, params)

    if (i + 1) % 10 == 0:
        energy = circuit(params)
        print(f"Step {i+1:3d} | Energy = {energy:.6f}")

Step  10 | Energy = -3.644248
Step  20 | Energy = -4.998744
Step  30 | Energy = -4.999999
Step  40 | Energy = -5.000000
Step  50 | Energy = -5.000000
Step  60 | Energy = -5.000000
Step  70 | Energy = -5.000000
Step  80 | Energy = -5.000000
Step  90 | Energy = -5.000000
Step 100 | Energy = -5.000000


In [17]:
# ============================================
# Final Results
# ============================================

final_energy = circuit(params)

print("\nOptimized Parameters:")
print(params)

print("\nGround-State Energy:")
print(final_energy)


Optimized Parameters:
[3.14159265 3.14159265]

Ground-State Energy:
-5.0


In [18]:
# ============================================
# Measure Most Probable State
# ============================================

@qml.qnode(dev)
def probabilities(params):

    ansatz(params)

    return qml.probs(wires=[0,1])

probs = probabilities(params)

print("\nProbabilities:")
print(probs)

states = ["00", "01", "10", "11"]

max_index = np.argmax(probs)

print(f"\nMost Probable Solution: {states[max_index]}")


Probabilities:
[7.79407322e-61 2.58605846e-32 1.00000000e+00 3.01388130e-29]

Most Probable Solution: 10
